# Home Credit：表内聚合（二）

## 1. `debitcard_1`

`debitcard_1` 是一张 repeated entity 长表，一行代表一条借记卡记录，`case_id + num_group1` 唯一定位一条记录。最终按 `case_id` 聚合为每个客户一行。

### 字段及处理方式

- **键字段**：`case_id` 用于分组和连接；`num_group1` 是记录编号。
- **数值字段**：3 个余额、流水字段，计算全部记录的 `count / sum / mean / min / max / std`，并补充缺失率。
- **日期字段**：`openingdate_857D` 是开户日期，计算最早日期、最晚日期、开户跨度及距申请日天数。

Train 中 97.11% 的客户不超过 3 条记录，因此额外按照开户日期保留最新 Top-3，补充整体统计无法保留的单张卡余额、流水和开户时间信息。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)

SPLIT = "train"       # 完成后改为 test 再运行一次
TOP_K = 3
SAVE_RESULT = False

ROOTS = [
    Path("/kaggle/input/competitions/home-credit-credit-risk-model-stability"),
    Path("/kaggle/input/home-credit-credit-risk-model-stability"),
]
DATA_ROOT = next((p for p in ROOTS if p.exists()), None)
assert DATA_ROOT is not None, "找不到比赛数据。"

PARQUET_ROOT = DATA_ROOT / "parquet_files"
TABLE_PATH = PARQUET_ROOT / SPLIT / f"{SPLIT}_debitcard_1.parquet"
BASE_PATH = PARQUET_ROOT / SPLIT / f"{SPLIT}_base.parquet"
OUTPUT_PATH = Path("/kaggle/working") / f"{SPLIT}_debitcard_1_agg.parquet"

In [ ]:
NUMERIC_COLS = [
    "last180dayaveragebalance_704A",
    "last180dayturnover_1134A",
    "last30dayturnover_651A",
]
DATE_COL = "openingdate_857D"
USE_COLS = ["case_id", "num_group1", *NUMERIC_COLS, DATE_COL]

debitcard = pd.read_parquet(TABLE_PATH, columns=USE_COLS)
base_dates = pd.read_parquet(BASE_PATH, columns=["case_id", "date_decision"])

debitcard[DATE_COL] = pd.to_datetime(debitcard[DATE_COL], errors="coerce")
base_dates["date_decision"] = pd.to_datetime(base_dates["date_decision"], errors="coerce")

assert not debitcard.duplicated(["case_id", "num_group1"]).any()
print("debitcard:", debitcard.shape)
display(debitcard.head())

### 定义聚合规则

定义数值、日期和最新 Top-3 记录的聚合方式。

In [ ]:
def aggregate_debitcard(df, base, top_k=3):
    df = df.copy()
    g = df.groupby("case_id", sort=False)

    # 全部记录统计
    record_count = g.size().rename("debitcard_record_count")
    stats = g[NUMERIC_COLS].agg(["count", "sum", "mean", "min", "max", "std"])
    stats.columns = [f"debitcard_{col}_{stat}" for col, stat in stats.columns]

    # 若该字段全部缺失，pandas 的 sum 会默认返回 0；改为 NaN，以区分“无数据”和“真实总和为 0”
    for col in NUMERIC_COLS:
        count_col = f"debitcard_{col}_count"
        sum_col = f"debitcard_{col}_sum"
        stats.loc[stats[count_col] == 0, sum_col] = np.nan
        stats[f"debitcard_{col}_missing_rate"] = 1 - stats[count_col] / record_count

    dates = g[DATE_COL].agg(["count", "nunique", "min", "max"])
    dates.columns = [
        "debitcard_openingdate_count",
        "debitcard_openingdate_nunique",
        "debitcard_earliest_openingdate",
        "debitcard_latest_openingdate",
    ]
    dates["debitcard_opening_span_days"] = (
    dates["debitcard_latest_openingdate"]
    - dates["debitcard_earliest_openingdate"]
).dt.days.astype("float64")

    # 最新 Top-K 记录, 按客户分组，并在每个客户内部按开户日期从新到旧排序；
    
    # 业务理由：最新开户的借记卡更能反映客户近期的账户状态；保留最新 Top-3 可覆盖 97.11% 的客户，
    # 在固定特征维度的同时，补充整体统计无法保留的单张卡余额、流水和开户时间信息。
    ranked = df.sort_values(
        ["case_id", DATE_COL, "num_group1"],
        ascending=[True, False, True],
        na_position="last",
    )
    top = ranked.groupby("case_id", sort=False).head(top_k).copy()
    top["rank"] = top.groupby("case_id", sort=False).cumcount() + 1
    top = top.pivot(index="case_id", columns="rank", values=[*NUMERIC_COLS, DATE_COL])
    top.columns = [f"debitcard_latest{rank}_{col}" for col, rank in top.columns]

    features = pd.concat([record_count, stats, dates, top], axis=1).reset_index()
    features["debitcard_has_record"] = 1
    features["debitcard_record_count_log1p"] = np.log1p(features["debitcard_record_count"])

    # 日期转换成相对申请日的天数
    features = features.merge(base, on="case_id", how="left", validate="one_to_one")
    features["debitcard_days_since_earliest_opening"] = (
        features["date_decision"] - features["debitcard_earliest_openingdate"]
    ).dt.days
    features["debitcard_days_since_latest_opening"] = (
        features["date_decision"] - features["debitcard_latest_openingdate"]
    ).dt.days

    raw_dates = ["debitcard_earliest_openingdate", "debitcard_latest_openingdate"]
    for rank in range(1, top_k + 1):
        col = f"debitcard_latest{rank}_{DATE_COL}"
        if col in features:
            features[f"debitcard_latest{rank}_days_since_opening"] = (
                features["date_decision"] - features[col]
            ).dt.days
            raw_dates.append(col)

    return features.drop(columns=["date_decision", *raw_dates])

### 执行聚合

运行聚合函数，检查结果并保存文件。

In [ ]:
debitcard_features = aggregate_debitcard(debitcard, base_dates, TOP_K)

assert debitcard_features["case_id"].is_unique
assert len(debitcard_features) == debitcard["case_id"].nunique()

print("聚合结果：", debitcard_features.shape)
display(debitcard_features.head())

if SAVE_RESULT:
    debitcard_features.to_parquet(OUTPUT_PATH, index=False)
    print("已保存：", OUTPUT_PATH)

## 2. `deposit_1`

`deposit_1` 是一张 repeated entity 长表，一行代表一个存款账户，`case_id + num_group1` 唯一定位一条记录。最终按 `case_id` 聚合为每个客户一行。

### 字段及处理方式

- **键字段**：`case_id` 用于分组和连接；`num_group1` 是账户记录编号。
- **数值字段**：`amount_416A` 是存款金额，计算全部账户的 `count / sum / mean / min / max / std`，并补充零值率和缺失率。
- **日期字段**：开户日期和合同结束日期用于计算开户距申请日天数、合同结束距申请日天数，以及合同持续时间。

Train 中 75.22% 的客户只有 1 条记录，97.43% 的客户不超过 3 条记录，因此额外按照开户日期保留最新 Top-3 账户，补充整体统计无法保留的单个账户信息。

In [ ]:
DEPOSIT_AMOUNT = "amount_416A"
DEPOSIT_OPEN_DATE = "openingdate_313D"
DEPOSIT_END_DATE = "contractenddate_991D"
DEPOSIT_COLS = [
    "case_id", "num_group1", DEPOSIT_AMOUNT,
    DEPOSIT_OPEN_DATE, DEPOSIT_END_DATE,
]

DEPOSIT_PATH = PARQUET_ROOT / SPLIT / f"{SPLIT}_deposit_1.parquet"
DEPOSIT_OUTPUT_PATH = Path("/kaggle/working") / f"{SPLIT}_deposit_1_agg.parquet"

deposit = pd.read_parquet(DEPOSIT_PATH, columns=DEPOSIT_COLS)
deposit[DEPOSIT_OPEN_DATE] = pd.to_datetime(deposit[DEPOSIT_OPEN_DATE], errors="coerce")
deposit[DEPOSIT_END_DATE] = pd.to_datetime(deposit[DEPOSIT_END_DATE], errors="coerce")

assert not deposit.duplicated(["case_id", "num_group1"]).any()
print("deposit:", deposit.shape)

### 定义聚合规则

聚合全部账户的金额、缺失情况和日期信息，并保留最新 Top-3 账户。

In [ ]:
def aggregate_deposit(df, base, top_k=3):
    df = df.merge(base, on="case_id", how="left", validate="many_to_one").copy()
    df["_days_since_opening"] = (df["date_decision"] - df[DEPOSIT_OPEN_DATE]).dt.days
    df["_days_until_contract_end"] = (df[DEPOSIT_END_DATE] - df["date_decision"]).dt.days
    df["_contract_duration_days"] = (df[DEPOSIT_END_DATE] - df[DEPOSIT_OPEN_DATE]).dt.days
    df["_amount_is_zero"] = df[DEPOSIT_AMOUNT].eq(0).where(df[DEPOSIT_AMOUNT].notna()).astype(float)

    g = df.groupby("case_id", sort=False)
    features = g.agg(
        deposit_record_count=("num_group1", "size"),
        deposit_amount_count=(DEPOSIT_AMOUNT, "count"),
        deposit_amount_sum=(DEPOSIT_AMOUNT, "sum"),
        deposit_amount_mean=(DEPOSIT_AMOUNT, "mean"),
        deposit_amount_min=(DEPOSIT_AMOUNT, "min"),
        deposit_amount_max=(DEPOSIT_AMOUNT, "max"),
        deposit_amount_std=(DEPOSIT_AMOUNT, "std"),
        deposit_amount_zero_rate=("_amount_is_zero", "mean"),
        deposit_openingdate_count=(DEPOSIT_OPEN_DATE, "count"),
        deposit_contractenddate_count=(DEPOSIT_END_DATE, "count"),
        deposit_days_since_opening_min=("_days_since_opening", "min"),
        deposit_days_since_opening_mean=("_days_since_opening", "mean"),
        deposit_days_since_opening_max=("_days_since_opening", "max"),
        deposit_days_until_contract_end_min=("_days_until_contract_end", "min"),
        deposit_days_until_contract_end_mean=("_days_until_contract_end", "mean"),
        deposit_days_until_contract_end_max=("_days_until_contract_end", "max"),
        deposit_contract_duration_days_mean=("_contract_duration_days", "mean"),
        deposit_contract_duration_days_min=("_contract_duration_days", "min"),
        deposit_contract_duration_days_max=("_contract_duration_days", "max"),
    )

    features.loc[features["deposit_amount_count"] == 0, "deposit_amount_sum"] = np.nan
    features["deposit_amount_missing_rate"] = 1 - features["deposit_amount_count"] / features["deposit_record_count"]
    features["deposit_openingdate_missing_rate"] = 1 - features["deposit_openingdate_count"] / features["deposit_record_count"]
    features["deposit_contractenddate_missing_rate"] = 1 - features["deposit_contractenddate_count"] / features["deposit_record_count"]

    ranked = df.sort_values(
        ["case_id", DEPOSIT_OPEN_DATE, "num_group1"],
        ascending=[True, False, True], na_position="last",
    )
    ranked["_rank"] = ranked.groupby("case_id").cumcount() + 1
    top_cols = {
        DEPOSIT_AMOUNT: "amount",
        "_days_since_opening": "days_since_opening",
        "_days_until_contract_end": "days_until_contract_end",
        "_contract_duration_days": "contract_duration_days",
    }
    for rank in range(1, top_k + 1):
        part = ranked.loc[ranked["_rank"] == rank, ["case_id", *top_cols]].set_index("case_id")
        part = part.rename(columns={col: f"deposit_latest{rank}_{name}" for col, name in top_cols.items()})
        features = features.join(part)

    features["deposit_has_record"] = 1
    features["deposit_record_count_log1p"] = np.log1p(features["deposit_record_count"])
    return features.reset_index()

### 执行聚合

运行聚合函数，检查结果并按需保存文件。

In [ ]:
deposit_features = aggregate_deposit(deposit, base_dates, TOP_K)

assert deposit_features["case_id"].is_unique
assert len(deposit_features) == deposit["case_id"].nunique()

print("聚合结果：", deposit_features.shape)
display(deposit_features.head())

if SAVE_RESULT:
    deposit_features.to_parquet(DEPOSIT_OUTPUT_PATH, index=False)
    print("已保存：", DEPOSIT_OUTPUT_PATH)

## 3. `other_1`

`other_1` 是一张 static/entity 表，每个 `case_id` 只有一行，因此不需要进行 `groupby` 聚合，可以直接作为客户级特征。

### 字段及处理方式

- **键字段**：`case_id` 用于连接；`num_group1` 恒为 0，只是结构编号，不作为模型特征。
- **数值字段**：5 个字段表示客户当前的交易金额或存款余额汇总，直接保留原始数值。
- **缺失与零值特征**：统计每个客户的有效值、缺失值和真实零值数量，并计算缺失率和零值比例，避免后续将“缺失”和“真实为 0”混为一类。

原始数值中的 0 表示没有相应交易或余额，与缺失值的含义不同，因此不将 0 替换为缺失值。

In [ ]:
OTHER_NUMERIC_COLS = [
    "amtdebitincoming_4809443A",
    "amtdebitoutgoing_4809440A",
    "amtdepositbalance_4809441A",
    "amtdepositincoming_4809444A",
    "amtdepositoutgoing_4809442A",
]
OTHER_COLS = ["case_id", "num_group1", *OTHER_NUMERIC_COLS]

OTHER_PATH = PARQUET_ROOT / SPLIT / f"{SPLIT}_other_1.parquet"
OTHER_OUTPUT_PATH = Path("/kaggle/working") / f"{SPLIT}_other_1_agg.parquet"

other = pd.read_parquet(OTHER_PATH, columns=OTHER_COLS)
assert other["case_id"].is_unique
print("other:", other.shape)
print("num_group1 unique:", other["num_group1"].dropna().unique())

### 整理字段

直接保留客户级数值，并统计每行包含多少缺失值和零值。

In [ ]:
def prepare_other(df):
    features = df[["case_id", *OTHER_NUMERIC_COLS]].copy()
    features = features.rename(columns={col: f"other_{col}" for col in OTHER_NUMERIC_COLS})
    value_cols = [f"other_{col}" for col in OTHER_NUMERIC_COLS]

    features["other_valid_count"] = features[value_cols].notna().sum(axis=1)
    features["other_missing_count"] = features[value_cols].isna().sum(axis=1)
    features["other_missing_rate"] = features["other_missing_count"] / len(value_cols)
    features["other_zero_count"] = features[value_cols].eq(0).sum(axis=1)
    features["other_zero_rate_among_valid"] = (
        features["other_zero_count"] / features["other_valid_count"].replace(0, np.nan)
    )
    features["other_has_record"] = 1
    return features

### 执行整理

生成每个客户一行的结果，检查后按需保存文件。

In [ ]:
other_features = prepare_other(other)

assert other_features["case_id"].is_unique
assert len(other_features) == len(other)

print("整理结果：", other_features.shape)
display(other_features.head())

if SAVE_RESULT:
    other_features.to_parquet(OTHER_OUTPUT_PATH, index=False)
    print("已保存：", OTHER_OUTPUT_PATH)

## 4. `person_1`

`person_1` 是一张 repeated entity 长表，一行代表一位与贷款申请相关的人员，`case_id + num_group1` 唯一定位一条人员记录。最终按 `case_id` 聚合为每个申请一行。

### 字段及处理方式

- **键字段**：`case_id` 用于分组和连接；`num_group1` 是人员记录编号。
- **数值字段**：保留主申请人的收入等数值，并为关联人员保留关键数值。
- **日期字段**：出生日期和入职日期分别转换为申请时的年龄和就业年限。
- **类别与布尔字段**：保留主申请人的人员类型、关系、角色及标记等完整信息，并为前 4 位关联人员保留关键字段。
- **结构特征**：统计总人数、关联人数及记录数的 `log1p`。

Train 中 50.39% 的申请只有 1 人，99.90% 的申请不超过 5 人，因此保留主申请人和前 4 位关联人员，在固定特征维度的同时覆盖绝大多数申请。

In [ ]:
PERSON_DATE_COLS = ["birth_259D", "birthdate_87D", "empl_employedfrom_271D"]
PERSON_RELATED_COLS = [
    "personindex_1023L", "persontype_1072L", "persontype_792L",
    "relationshiptoclient_415T", "relationshiptoclient_642T",
    "role_1084L", "role_993L", "type_25L",
    "isreference_387L", "remitter_829L", "mainoccupationinc_384A",
]
PERSON_RELATED_TOP_K = 4

PERSON_PATH = PARQUET_ROOT / SPLIT / f"{SPLIT}_person_1.parquet"
PERSON_OUTPUT_PATH = Path("/kaggle/working") / f"{SPLIT}_person_1_agg.parquet"

person = pd.read_parquet(PERSON_PATH)
for col in PERSON_DATE_COLS:
    person[col] = pd.to_datetime(person[col], errors="coerce")

assert not person.duplicated(["case_id", "num_group1"]).any()
print("person:", person.shape)

### 定义聚合规则

将第一位人员作为主申请人，其余人员按顺序保留前 4 位；日期转换为申请时的年龄和就业年限。

In [ ]:
def aggregate_person(df, base, related_top_k=4):
    df = df.merge(base, on="case_id", how="left", validate="many_to_one").copy()
    birth_date = df["birth_259D"].combine_first(df["birthdate_87D"])
    df["age_years"] = (df["date_decision"] - birth_date).dt.days / 365.25
    df["employment_years"] = (df["date_decision"] - df["empl_employedfrom_271D"]).dt.days / 365.25

    df = df.sort_values(["case_id", "num_group1", "personindex_1023L"], na_position="last")
    df["_person_rank"] = df.groupby("case_id").cumcount() + 1
    record_count = df.groupby("case_id").size().rename("person_record_count")

    excluded = {"case_id", "num_group1", "date_decision", "_person_rank", *PERSON_DATE_COLS}
    main_cols = [col for col in df.columns if col not in excluded]
    main = df.loc[df["_person_rank"] == 1, ["case_id", *main_cols]].set_index("case_id")
    main = main.rename(columns={col: f"person_main_{col}" for col in main_cols})

    features = record_count.to_frame().join(main)
    related_cols = [*PERSON_RELATED_COLS, "age_years", "employment_years"]
    for rank in range(1, related_top_k + 1):
        part = df.loc[df["_person_rank"] == rank + 1, ["case_id", *related_cols]].set_index("case_id")
        part = part.rename(columns={col: f"person_related{rank}_{col}" for col in related_cols})
        features = features.join(part)

    features["person_related_count"] = (features["person_record_count"] - 1).clip(lower=0)
    features["person_record_count_log1p"] = np.log1p(features["person_record_count"])
    features["person_has_record"] = 1
    return features.reset_index()

### 执行聚合

生成每个客户一行的结果，检查后按需保存文件。

In [ ]:
person_features = aggregate_person(person, base_dates, PERSON_RELATED_TOP_K)

assert person_features["case_id"].is_unique
assert len(person_features) == person["case_id"].nunique()

print("聚合结果：", person_features.shape)
display(person_features.head())

if SAVE_RESULT:
    person_features.to_parquet(PERSON_OUTPUT_PATH, index=False)
    print("已保存：", PERSON_OUTPUT_PATH)